In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist
import biotite.structure as struc
import biotite.structure.io as strucio
import biotite.sequence as seq
import biotite.sequence.align as align

# Standard mapping for 3-letter to 1-letter amino acid codes
AA_MAP = {
  'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C', 'GLN': 'Q',
  'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I', 'LEU': 'L', 'LYS': 'K',
  'MET': 'M', 'PHE': 'F', 'PRO': 'P', 'SER': 'S', 'THR': 'T', 'TRP': 'W',
  'TYR': 'Y', 'VAL': 'V'
}

def _load_and_extract_chain(raw_id, cif_dir="fixed_cif_files"):
  """Helper to load a structure and resolve the entity/chain ID."""
 
  source = os.path.join(cif_dir, f"{raw_id}.cif")
  
  if not source:
    raise FileNotFoundError(f"Missing file for {raw_id}")

  array = strucio.load_structure(source)
  return array[struc.filter_amino_acids(array)]

def _get_ca_sequence_and_coords(array):
  """Extracts the 1-letter sequence, residue IDs, and CA coordinates."""
  ca_array = array[array.atom_name == 'CA']
  
  # Remove alternates to prevent duplicates
  if "altloc_id" in ca_array.get_annotation_categories():
    ca_array = ca_array[(ca_array.altloc_id == '') | (ca_array.altloc_id == 'A') | (ca_array.altloc_id == '1')]
    
  seq_str = "".join([AA_MAP.get(name, 'X') for name in ca_array.res_name])
  return seq_str, ca_array.res_id, ca_array.coord

def compare_fold_switch_contacts(id1, id2, threshold=8.0, cif_dir="fixed_cif_files", sse_csv="residue_switch_labels.csv", plot_heatmap=True):
  """
  Creates contact maps for two conformations, performs local sequence alignment,
  and analyzes the change in distance for contacts that differ between them.
  """
  print(f"Loading structures {id1} and {id2}...")
  try:
    arr1 = _load_and_extract_chain(id1, cif_dir)
    arr2 = _load_and_extract_chain(id2, cif_dir)
  except FileNotFoundError as e:
    print(f"Error: {e}")
    return

  seq1_str, res_ids1, coords1 = _get_ca_sequence_and_coords(arr1)
  seq2_str, res_ids2, coords2 = _get_ca_sequence_and_coords(arr2)

  print(f"Extracted Sequences:\n{id1}: Length {len(seq1_str)}\n{id2}: Length {len(seq2_str)}")

  # Fetch fold-switch labels from CSV
  try:
    labels_df = pd.read_csv(sse_csv)
    df1 = labels_df[labels_df['Raw_Input_ID'] == id1]
    # A residue is fold-switching if switch_SS_label != 'x' and SS_Label != switch_SS_label
    df1_fs = df1[(df1['switch_SS_label'] != 'x') & (df1['SS_Label'] != df1['switch_SS_label'])]
    fs_dict = {}
    for _, row in df1_fs.iterrows():
      fs_dict[row['Residue_ID']] = f"{row['SS_Label']}->{row['switch_SS_label']}"
  except Exception as e:
    print(f"Error reading switch labels from {sse_csv}: {e}")
    fs_dict = {}

  # --- 1. Perform Pairwise Local Alignment ---
  print("Performing Pairwise Local Alignment...")
  matrix = align.SubstitutionMatrix.std_protein_matrix()
  seq1 = seq.ProteinSequence(seq1_str)
  seq2 = seq.ProteinSequence(seq2_str)
  
  alignments = align.align_optimal(seq1, seq2, matrix, local=True)
  if not alignments:
    print("No valid local alignment found.")
    return
    
  best_alignment = alignments[0]
  
  # Extract valid aligned indices (ignore gaps represented by -1)
  trace = best_alignment.trace
  valid_mask = (trace[:, 0] != -1) & (trace[:, 1] != -1)
  idx1_aligned = trace[valid_mask, 0]
  idx2_aligned = trace[valid_mask, 1]
  
  num_aligned = len(idx1_aligned)
  print(f"Successfully aligned {num_aligned} residues.")

  # Filter coordinates and residue IDs to only the aligned positions
  c1_aln = coords1[idx1_aligned]
  c2_aln = coords2[idx2_aligned]
  r1_aln = res_ids1[idx1_aligned]
  r2_aln = res_ids2[idx2_aligned]

  # --- 2. Create Contact Maps ---
  # Calculate pairwise distance matrices for the aligned CA atoms
  dist1 = cdist(c1_aln, c1_aln)
  dist2 = cdist(c2_aln, c2_aln)
  
  # Threshold into boolean contact maps
  cmap1 = dist1 < threshold
  cmap2 = dist2 < threshold

  # --- 3. Identify Differences ---
  # Find contacts that exist in one map but not the other
  diff_mask = cmap1 != cmap2
  
  # Mask out the lower triangle and diagonal to avoid double counting
  diff_mask = np.triu(diff_mask, k=1)
  
  if not np.any(diff_mask):
    print(f"\nNo differing contacts found at a {threshold}Å threshold!")
    return None, None, None

  # Calculate the absolute distance change for differing contacts
  dist_changes = np.abs(dist1 - dist2)
  diff_changes_1d = dist_changes[diff_mask]
  
  # Find the maximum distance change
  max_idx_1d = np.argmax(diff_changes_1d)
  max_change = diff_changes_1d[max_idx_1d]
  
  # Map back to 2D indices
  i_aln, j_aln = np.argwhere(diff_mask)[max_idx_1d]
  
  # Check fold-switching status for global max
  res_i, res_j = r1_aln[i_aln], r1_aln[j_aln]
  status_i = f"Yes ({fs_dict[res_i]})" if res_i in fs_dict else "No"
  status_j = f"Yes ({fs_dict[res_j]})" if res_j in fs_dict else "No"

  # --- Find max change specifically amongst fold-switching positions ---
  is_fs_aln = np.array([res in fs_dict for res in r1_aln])
  fs_mask_2d = is_fs_aln[:, None] & is_fs_aln[None, :]
  fs_diff_mask = diff_mask & fs_mask_2d
  
  i_fs, j_fs = None, None
  fs_max_change = None
  
  if np.any(fs_diff_mask):
    fs_diff_changes_1d = dist_changes[fs_diff_mask]
    fs_max_idx_1d = np.argmax(fs_diff_changes_1d)
    fs_max_change = fs_diff_changes_1d[fs_max_idx_1d]
    i_fs, j_fs = np.argwhere(fs_diff_mask)[fs_max_idx_1d]
    res_i_fs, res_j_fs = r1_aln[i_fs], r1_aln[j_fs]
    fs_status_i = fs_dict[res_i_fs]
    fs_status_j = fs_dict[res_j_fs]
  
  print("\n=========================================================")
  print(f" FOLD SWITCH CONTACT ANALYSIS (Threshold: {threshold}Å)")
  print("=========================================================")
  print(f"Total Aligned Positions: {num_aligned}")
  print(f"Total Differing Contacts Found: {np.sum(diff_mask)}")
  print(f"\n-> GLOBAL MAXIMUM DISTANCE CHANGE:")
  print(f"   Change: {max_change:.2f} Å")
  print(f"   {id1} Residues: {res_i} [FS: {status_i}] and {res_j} [FS: {status_j}] (Dist: {dist1[i_aln, j_aln]:.2f} Å)")
  print(f"   {id2} Residues: {r2_aln[i_aln]} and {r2_aln[j_aln]} (Dist: {dist2[i_aln, j_aln]:.2f} Å)")
  
  if i_fs is not None:
    print(f"\n-> MAXIMUM DISTANCE CHANGE (Amongst Fold-Switching Positions):")
    print(f"   Change: {fs_max_change:.2f} Å")
    print(f"   {id1} Residues: {res_i_fs} [FS: {fs_status_i}] and {res_j_fs} [FS: {fs_status_j}] (Dist: {dist1[i_fs, j_fs]:.2f} Å)")
    print(f"   {id2} Residues: {r2_aln[i_fs]} and {r2_aln[j_fs]} (Dist: {dist2[i_fs, j_fs]:.2f} Å)")
  else:
    print(f"\n-> MAXIMUM DISTANCE CHANGE (Amongst Fold-Switching Positions):")
    print("   No differing contacts found where BOTH residues are fold-switching.")
  print("=========================================================\n")

  fs_contact_type = None
  if i_fs is not None and j_fs is not None:
    t_i = fs_dict.get(r1_aln[i_fs], "Unknown")
    t_j = fs_dict.get(r1_aln[j_fs], "Unknown")
    # Cleanly format the contact type, e.g. "a->b & a->b" or "a->b / c->b"
    fs_contact_type = f"{t_i} & {t_j}" if t_i == t_j else " / ".join(sorted([t_i, t_j]))

  if not plot_heatmap:
    return max_change, fs_max_change, fs_contact_type
    
  # --- 4. Visualizing the Differences ---
  plt.figure(figsize=(11, 8))
  
  # Create a masked array so we only color the changed contacts
  viz_matrix = np.full_like(dist_changes, np.nan)
  viz_matrix[diff_mask | diff_mask.T] = dist_changes[diff_mask | diff_mask.T]
  
  sns.heatmap(
    viz_matrix, 
    cmap='magma_r', 
    cbar_kws={'label': 'Absolute Distance Change (Å)'},
    xticklabels=10, 
    yticklabels=10,
    square=True
  )
  
  plt.title(f'Structural Shifts in Altered Contacts\n{id1} vs {id2}', fontsize=14, fontweight='bold', pad=15)
  plt.xlabel(f'Aligned Residue Index', fontsize=12)
  plt.ylabel(f'Aligned Residue Index', fontsize=12)
  
  # Highlight the maximum change coordinate on the plot
  label_str = 'Global Max Change'
  if i_fs is not None and i_fs == i_aln and j_fs == j_aln:
    label_str = 'Global & FS Max Change'
    
  plt.plot(j_aln, i_aln, marker='*', color='cyan', markersize=12, markeredgecolor='black', label=label_str)
  plt.plot(i_aln, j_aln, marker='*', color='cyan', markersize=12, markeredgecolor='black') # Symmetry
  
  # Highlight the FS max change if it exists and is different from global max
  if i_fs is not None and (i_fs != i_aln or j_fs != j_aln):
    plt.plot(j_fs, i_fs, marker='o', color='lime', markersize=10, markeredgecolor='black', label='FS Max Change')
    plt.plot(i_fs, j_fs, marker='o', color='lime', markersize=10, markeredgecolor='black')
      
  # Move legend completely outside the plot area
  plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
  
  plt.tight_layout()
  plt.show()


In [3]:
import pandas as pd
meta_df = pd.read_csv('metamorphic_pairs.csv')

In [ ]:
for row in meta_df.iterrows():
  pdb1, pdb2 = row[1]['PDB1'], row[1]['PDB2']
  pdb1 = pdb1[:-1].upper()+'.'+pdb1[-1]
  pdb2 = pdb2[:-1].upper()+'.'+pdb2[-1]

  print(f"\n --------- {pdb1}/{pdb2}")
  compare_fold_switch_contacts(pdb1, pdb2)

In [4]:
def plot_batch_distance_changes(meta_csv_path='metamorphic_pairs.csv', threshold=8.0, cif_dir="fixed_cif_files", sse_csv="residue_switch_labels.csv"):
  """
  Reads a metadata CSV containing 'PDB1' and 'PDB2' columns, computes maximum 
  distance changes for each pair, and generates subplots mapping them out.
  """
  print(f"Loading metadata from {meta_csv_path}...")
  try:
    meta_df = pd.read_csv(meta_csv_path)
  except FileNotFoundError:
    print(f"Error: {meta_csv_path} not found.")
    return

  if 'PDB1' not in meta_df.columns or 'PDB2' not in meta_df.columns:
    print("Error: CSV must contain 'PDB1' and 'PDB2' columns.")
    return

  labels = []
  global_changes = []
  fs_changes = []
  fs_types = []

  # Iterate exactly as requested
  for _, row in meta_df.iterrows():
    raw_pdb1, raw_pdb2 = str(row['PDB1']).strip(), str(row['PDB2']).strip()
    
    # Apply the string formatting (e.g., '2KB8A' -> '2KB8.A')
    if len(raw_pdb1) >= 5 and '.' not in raw_pdb1 and '_' not in raw_pdb1:
      pdb1 = raw_pdb1[:-1].upper() + '.' + raw_pdb1[-1].upper()
    else:
      pdb1 = raw_pdb1.upper()
      
    if len(raw_pdb2) >= 5 and '.' not in raw_pdb2 and '_' not in raw_pdb2:
      pdb2 = raw_pdb2[:-1].upper() + '.' + raw_pdb2[-1].upper()
    else:
      pdb2 = raw_pdb2.upper()

    print(f"\n --------- {pdb1} / {pdb2} ---------")
    
    max_c, fs_max_c, fs_type = compare_fold_switch_contacts(
      pdb1, pdb2, threshold=threshold, 
      cif_dir=cif_dir, sse_csv=sse_csv, 
      plot_heatmap=False  # Disable heatmap for loop
    )
    
    labels.append(f"{pdb1}/{pdb2}")
    global_changes.append(max_c if max_c is not None else np.nan)
    fs_changes.append(fs_max_c if fs_max_c is not None else np.nan)
    fs_types.append(fs_type)

  # --- Matplotlib Rendering for Batch Data ---
  # Create a figure with 2 rows and 1 column, sharing the X-axis
  fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
  
  x_indices = np.arange(len(labels))
  
  # Top Plot: Global Max Change
  ax1.plot(x_indices, global_changes, marker='o', color='tab:blue', linestyle='-', linewidth=2, markersize=8)
  ax1.set_title('Global Maximum Distance Change per PDB Pair', fontsize=14, fontweight='bold')
  ax1.set_ylabel('Max Distance Change (Å)', fontsize=12)
  ax1.grid(True, alpha=0.4, linestyle='--')
  
  # Bottom Plot: Fold-Switch Specific Max Change
  # Draw the underlying connecting line first
  ax2.plot(x_indices, fs_changes, color='gray', linestyle='-', linewidth=2, alpha=0.4, zorder=1)
  
  # Map unique fold-switch contact types to styles
  unique_types = sorted(list(set([t for t in fs_types if t is not None])))
  markers = ['s', '^', 'D', 'v', 'p', '*', 'h', 'o', '<', '>']
  colors = plt.cm.Set2.colors if len(unique_types) <= 8 else plt.cm.tab20.colors
  
  # Scatter plot for each distinct fold-switch contact type
  for i, u_type in enumerate(unique_types):
    mask = [t == u_type for t in fs_types]
    x_t = x_indices[mask]
    y_t = np.array(fs_changes)[mask]
    
    ax2.scatter(
      x_t, y_t, 
      marker=markers[i % len(markers)], 
      color=colors[i % len(colors)], 
      edgecolor='black', 
      s=120, 
      zorder=2, 
      label=u_type
    )

  ax2.set_title('Maximum Distance Change (Strictly Amongst Fold-Switching Residues)', fontsize=14, fontweight='bold')
  ax2.set_ylabel('FS Max Change (Å)', fontsize=12)
  ax2.set_xlabel('PDB Pairs', fontsize=12)
  ax2.grid(True, alpha=0.4, linestyle='--')
  
  # Format X-axis
  ax2.set_xticks(x_indices)
  ax2.set_xticklabels(labels, rotation=90, ha='center', fontsize=10)
  
  if unique_types:
      ax2.legend(title='Fold-Switch\nContact Type', bbox_to_anchor=(1.01, 1), loc='upper left')
  
  plt.tight_layout()
  plt.show()

In [ ]:
plot_batch_distance_changes()

In [5]:
from matplotlib.patches import Patch, Rectangle
def display_alignment_with_sse(id1, id2, cif_dir="fixed_cif_files", chunk_size=60, sse_csv="residue_secondary_structures.csv"):
  """
  Performs pairwise local alignment of two sequences and displays the aligned 
  sequences alongside their secondary structure annotations and original residue positions.
  """
  print(f"Loading structures {id1} and {id2}...")
  try:
    arr1 = _load_and_extract_chain(id1, cif_dir)
    arr2 = _load_and_extract_chain(id2, cif_dir)
  except FileNotFoundError as e:
    print(f"Error: {e}")
    return

  seq1_str, res_ids1, _ = _get_ca_sequence_and_coords(arr1)
  seq2_str, res_ids2, _ = _get_ca_sequence_and_coords(arr2)

  # Fetch SSE from CSV
  try:
    sse_df = pd.read_csv(sse_csv)
    
    df1 = sse_df[sse_df['Raw_Input_ID'] == id1]
    sse1_dict = dict(zip(df1['Residue_ID'], df1['SS_Label']))
    
    df2 = sse_df[sse_df['Raw_Input_ID'] == id2]
    sse2_dict = dict(zip(df2['Residue_ID'], df2['SS_Label']))
  except Exception as e:
    print(f"Error reading SSE from {sse_csv}: {e}")
    sse1_dict = {}
    sse2_dict = {}
    
  sse1 = [sse1_dict.get(r, 'c') for r in res_ids1]
  sse2 = [sse2_dict.get(r, 'c') for r in res_ids2]

  print("Performing Pairwise Local Alignment...")
  matrix = align.SubstitutionMatrix.std_protein_matrix()
  seq1 = seq.ProteinSequence(seq1_str)
  seq2 = seq.ProteinSequence(seq2_str)
  
  alignments = align.align_optimal(seq1, seq2, matrix, local=True)
  if not alignments:
    print("No valid local alignment found.")
    return
    
  best_alignment = alignments[0]
  trace = best_alignment.trace
  
  id1_pad = id1[:8].ljust(8)
  id2_pad = id2[:8].ljust(8)
  
  # --- Graphical Matplotlib Rendering ---
  num_chunks = (len(trace) + chunk_size - 1) // chunk_size
  chunk_height = 4.5
  total_lines = (num_chunks * chunk_height) + 3
  
  fig, ax = plt.subplots(figsize=(14, max(5, num_chunks * 1.8)))
  ax.set_ylim(0, total_lines)
  ax.set_xlim(-15, chunk_size + 5)
  ax.axis('off')
  
  fig.suptitle(f"SEQUENCE ALIGNMENT & SECONDARY STRUCTURE: {id1} vs {id2}", 
               family='monospace', weight='bold', fontsize=16, y=0.98)
  
  # Added legend positioned to the upper right (including Mutation flag)
  legend_elements = [
    Patch(facecolor='#ff4c4c', label=r'$\alpha$-Helix'),
    Patch(facecolor='#4c4cff', label=r'$\beta$-Sheet'),
    Patch(facecolor='#cccccc', label='Coil'),
    Patch(facecolor='#ffcdd2', label='Mutation')
  ]
  ax.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.0, 1.0), 
            ncol=1, frameon=True, fontsize=10)
  
  sse_colors = {'a': '#ff4c4c', 'b': '#4c4cff', 'c': '#cccccc', ' ': 'none'}
  y_pos = total_lines - 4.0
  
  for i in range(0, len(trace), chunk_size):
    chunk = trace[i:i+chunk_size]
    
    s1_chars, sse1_chars = [], []
    s2_chars, sse2_chars = [], []
    
    start1, end1 = None, None
    start2, end2 = None, None
    
    for idx1, idx2 in chunk:
      if idx1 != -1:
        s1_chars.append(seq1_str[idx1])
        sse1_chars.append(sse1[idx1])
        if start1 is None: start1 = res_ids1[idx1]
        end1 = res_ids1[idx1]
      else:
        s1_chars.append('-')
        sse1_chars.append(' ')
        
      if idx2 != -1:
        s2_chars.append(seq2_str[idx2])
        sse2_chars.append(sse2[idx2])
        if start2 is None: start2 = res_ids2[idx2]
        end2 = res_ids2[idx2]
      else:
        s2_chars.append('-')
        sse2_chars.append(' ')
        
    start1_str = str(start1) if start1 is not None else ""
    end1_str = str(end1) if end1 is not None else ""
    start2_str = str(start2) if start2 is not None else ""
    end2_str = str(end2) if end2 is not None else ""
    
    ax.text(-1, y_pos + 2, f"{id1_pad} {start1_str:<4}", family='monospace', va='center', ha='right', color='black', weight='bold')
    ax.text(-1, y_pos + 1, f"{id2_pad} {start2_str:<4}", family='monospace', va='center', ha='right', color='black', weight='bold')
    
    ax.text(len(chunk) + 1, y_pos + 2, end1_str, family='monospace', va='center', ha='left', color='black', weight='bold')
    ax.text(len(chunk) + 1, y_pos + 1, end2_str, family='monospace', va='center', ha='left', color='black', weight='bold')
    
    for j in range(len(chunk)):
      # Determine if a mutation (substitution) exists at this position
      is_mutation = (s1_chars[j] != s2_chars[j]) and (s1_chars[j] != '-') and (s2_chars[j] != '-')
      
      # Draw soft red column background highlight if mutation is present
      if is_mutation:
        rect_mut = Rectangle((j - 0.5, y_pos + 0.7), 1.0, 1.6, facecolor='#ffcdd2', edgecolor='none', zorder=1)
        ax.add_patch(rect_mut)
        text_color = '#c62828' # Pop text out in red
      else:
        text_color = 'black'
        
      # Write character sequences on top of background highlights
      ax.text(j, y_pos + 2, s1_chars[j], family='monospace', va='center', ha='center', fontsize=11, weight='bold', color=text_color, zorder=3)
      ax.text(j, y_pos + 1, s2_chars[j], family='monospace', va='center', ha='center', fontsize=11, weight='bold', color=text_color, zorder=3)
      
      if s1_chars[j] == s2_chars[j] and s1_chars[j] != '-':
        ax.text(j, y_pos + 1.5, '|', family='monospace', va='center', ha='center', fontsize=10, color='gray', zorder=3)
      
      color1 = sse_colors.get(sse1_chars[j], 'none')
      if color1 != 'none':
        rect1 = Rectangle((j - 0.5, y_pos + 2.5), 1.0, 0.4, facecolor=color1, edgecolor='none', zorder=2)
        ax.add_patch(rect1)
          
      color2 = sse_colors.get(sse2_chars[j], 'none')
      if color2 != 'none':
        rect2 = Rectangle((j - 0.5, y_pos + 0.1), 1.0, 0.4, facecolor=color2, edgecolor='none', zorder=2)
        ax.add_patch(rect2)
        
    y_pos -= chunk_height
    
  plt.tight_layout(rect=[0, 0, 1, 0.95])
  plt.show()

In [ ]:
for row in meta_df.iterrows():
  pdb1, pdb2 = row[1]['PDB1'], row[1]['PDB2']
  pdb1 = pdb1[:-1].upper()+'.'+pdb1[-1]
  pdb2 = pdb2[:-1].upper()+'.'+pdb2[-1]

  print(f"\n --------- {pdb1}/{pdb2}")
  display_alignment_with_sse(pdb1, pdb2)

In [6]:
import biotite.structure.io.pdb as pdb
import biotite.structure.io.pdbx as pdbx
import biotite.database.rcsb as rcsb
import pandas as pd
def get_model_counts(pdb_ids):
  model_counts = {}
  
  for pdb_id in pdb_ids:
    try:      
      # file_path = f"cif_files/{pdb_id}.cif"
      file_path = rcsb.fetch(pdb_id, "cif")
      # 2. Read the file
      # cif_file = pdbx.CIFFile.read(file_path)
      cif_file = pdbx.CIFFile.read(file_path)
            
      # 3. Extract the number of models
      num_models = pdbx.get_model_count(cif_file)
      model_counts[pdb_id] = num_models
        
    except Exception as e:
      model_counts[pdb_id] = f"Error fetching or parsing: {e}"
          
  return model_counts

In [ ]:
pdb_df = pd.read_csv('pdb_secondary_structures.csv')
pdb_ids = []
for row in pdb_df.iterrows():
  pdbid = row[1]['Raw_Input_ID'][:4]
  pdb_ids.append(pdbid)
model_counts = get_model_counts(pdb_ids)

In [ ]:
model_counts.values()

In [ ]:
len(model_counts), len(pdb_df), len(pdb_ids), pdb_ids

In [9]:
display_alignment_with_sse('1MNM.C', '1MNM.D')